In [ ]:
from google.colab import files
import os, shutil

# Upload: key_numbers.json, summary_monthly.csv, summary_segments.csv,
#         summary_country.csv, summary_weekday.csv, summary_product.csv
os.makedirs("output", exist_ok=True)
for name in files.upload():
    shutil.move(name, f"output/{name}")
print(os.listdir("output"))

In [ ]:
import json, pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
R = json.load(open("output/key_numbers.json"))
f = lambda x: f"GBP {x:,.0f}"; p = lambda x: f"{x*100:.1f}%"
wb = Workbook(); wb.remove(wb.active)
H = PatternFill("solid", fgColor="1F7A8C"); thin = Side(style="thin", color="BBBBBB")
def sheet(name, widths):
    ws = wb.create_sheet(name)
    for i,w in enumerate(widths,1): ws.column_dimensions[get_column_letter(i)].width = w
    return ws
def put(ws, rows, header_rows=(1,)):
    for r in rows: ws.append(r)
    for row in ws.iter_rows():
        for c in row:
            c.font = Font(name="Arial", size=10); c.alignment = Alignment(wrap_text=True, vertical="top")
            if c.value is not None: c.border = Border(top=thin,bottom=thin,left=thin,right=thin)
    for hr in header_rows:
        for c in ws[hr]:
            if c.value is not None: c.font = Font(name="Arial", size=10, bold=True, color="FFFFFF"); c.fill = H
def title(ws, t): ws.append([t]); ws["A1"].font = Font(name="Arial", size=13, bold=True)

In [ ]:
# Supporting tabs
def csvsheet(name, path, cols=None, fmt=None, limit=None):
    d = pd.read_csv(path); d = d.head(limit) if limit else d
    ws = sheet(name, [22]+[16]*(len(d.columns)-1)); ws.append(list(d.columns))
    for r in d.itertuples(index=False): ws.append([None if pd.isna(x) else (round(x,4) if isinstance(x,float) else x) for x in r])
    for row in ws.iter_rows():
        for c in row: c.font = Font(name="Arial", size=10)
    for c in ws[1]: c.font = Font(name="Arial", size=10, bold=True, color="FFFFFF"); c.fill = H
csvsheet("Monthly", "output/summary_monthly.csv"); csvsheet("Segments", "output/summary_segments.csv")
csvsheet("Country", "output/summary_country.csv"); csvsheet("Weekday", "output/summary_weekday.csv")
csvsheet("Top Products", "output/summary_product.csv", limit=100)
order = ["Start Here","Q1","Q2","Q3","Q4","Q5","Q6","Q7","Q8","Q10","Monthly","Segments","Country","Weekday","Top Products"]
wb._sheets = [wb[n] for n in order]; wb.save("VirtuBox_Assessment_Workbook.xlsx"); print("saved")

In [ ]:
files.download("VirtuBox_Assessment_Workbook.xlsx")